# 10장 — 기억의 경제학: 컨텍스트 예산

웹 챕터(10장)와 **같은 카탈로그, 같은 수치**를 파이썬으로 직접 계산합니다.
웹 실험실(ContextLab)의 냅색 게임을 **전수 탐색**으로 풀어 최적 조합이 정말 최고 점수인지 검증해요.

> 🤖 **AI 튜터와 함께 공부하세요** — 아래 셀을 실행하면 화면 오른쪽 아래에 튜터 버튼이 뜹니다.
> 로컬 Ollama 가 꺼져 있어도 노트북 학습에는 지장이 없습니다.

In [ ]:
from tutor import chat
chat('ch10')  # 이 챕터를 아는 AI 튜터 (로컬 Ollama — 없어도 학습엔 지장 없음)

## 1. 카탈로그 — 무엇을 책상에 올릴까

예산(컨텍스트 윈도)은 **8,000토큰**. 각 항목은 (토큰, 기여 점수)를 가집니다.
필수 항목(시스템 규칙·과업 설명)은 항상 올라가고, 궤적은 원본/요약/제외 중 하나를 고릅니다.

In [ ]:
BUDGET = 8000

# 필수 (항상 포함) — 토큰 합 1000, 점수 합 33
MANDATORY_TOK, MANDATORY_SCORE = 600 + 400, 15 + 18

# 궤적: 셋 중 하나 (원본/요약/제외)
TRAJ = {'full': (3600, 20), 'summary': (900, 14), 'none': (0, 0)}

# 선택 항목 (담거나 말거나)
OPTS = {
    'A':    (1500, 16),  # 관련 문서 A (핵심 사양서)
    'B':    (1500, 9),   # 관련 문서 B (참고 지침)
    'C':    (1500, 4),   # 관련 문서 C (관련성 낮음)
    'E1':   (1200, 11),  # 모범 예시 1개
    'E2':   (1200, 5),   # 예시 2개째 (수확 체감)
    'JUNK': (800, -3),   # 잡동사니 — 지난 잡담 로그 (방해!)
}

def density(tok, score):
    return score / tok * 1000  # 점 / 1k토큰

print('밀도(점/1k토큰) 순위:')
rows = [('궤적요약', *TRAJ['summary']), ('궤적원본', *TRAJ['full'])] + [(k, *v) for k, v in OPTS.items()]
for name, tok, sc in sorted(rows, key=lambda r: -density(r[1], r[2])):
    print(f'  {name:8s} {tok:5d}tk  {sc:+3d}  밀도 {density(tok, sc):5.1f}')

## 2. 시작 상태 — 다 끌고 다니는 습관

도움이 될까 싶어 전부 올리면(원본 궤적 + 문서 A·B + 예시 + 잡동사니) **예산 초과**가 됩니다.

In [ ]:
def config_cost(traj, picks):
    tok = MANDATORY_TOK + TRAJ[traj][0] + sum(OPTS[k][0] for k in picks)
    score = MANDATORY_SCORE + TRAJ[traj][1] + sum(OPTS[k][1] for k in picks)
    return tok, score

start_tok, start_score = config_cost('full', ['A', 'B', 'E1', 'JUNK'])
print(f'시작(원본+A+B+예시+잡동사니): {start_tok:,}토큰 / 점수 {start_score}')
print(f'예산 8,000 초과? {start_tok > BUDGET}  (초과분 {start_tok - BUDGET:,}토큰)')
assert start_tok == 9600  # 다 끌고 다니면 9,600토큰 — 책상이 넘친다
assert start_tok > BUDGET

## 3. 전수 탐색 — 진짜 최적은?

궤적 3가지 × 선택 항목 2⁶ = 192가지 조합을 전부 훑어 **예산 안에서 최고 점수**를 찾습니다.
이것이 배낭 문제(knapsack) — 항목 수가 작으니 전수 탐색으로 정확한 최적을 확정할 수 있어요.

In [ ]:
from itertools import combinations

keys = list(OPTS.keys())
best = None
for traj in TRAJ:
    for r in range(len(keys) + 1):
        for picks in combinations(keys, r):
            tok, score = config_cost(traj, picks)
            if tok <= BUDGET and (best is None or score > best[0]):
                best = (score, tok, traj, picks)

best_score, best_tok, best_traj, best_picks = best
print(f'최적: 궤적={best_traj} + {sorted(best_picks)}')
print(f'  {best_tok:,}토큰 / {best_score}점')

# 웹 실험실(ContextLab)의 '최적 조합 보기'와 일치하는지 검증
assert best_traj == 'summary'
assert set(best_picks) == {'A', 'B', 'E1', 'E2'}
assert best_tok == 7300 and best_score == 88
print('\n✅ 웹 실험실의 최적(요약 + A + B + E1 + E2 = 88점)과 일치')

## 4. 요약이라는 거래 + 탐욕이 통하는지

원본 궤적을 요약으로 바꾸면 2,700토큰이 열리고, 그 자리에 밀도 높은 항목을 담아 순증을 만듭니다.
그리고 '밀도 높은 것부터 담기(탐욕)'가 이번 예제에서 최적과 일치하는지도 확인합니다.

In [ ]:
# 요약이라는 거래 — StepQuiz 와 같은 계산
freed = TRAJ['full'][0] - TRAJ['summary'][0]
refill = OPTS['A'][0] + OPTS['E1'][0]
net = (TRAJ['summary'][1] - TRAJ['full'][1]) + OPTS['A'][1] + OPTS['E1'][1]
print(f'요약으로 열린 공간: {freed:,}토큰,  A+예시 크기: {refill:,}토큰 (딱 맞음: {freed == refill})')
print(f'순 점수 변화: (14-20) + 16 + 11 = {net:+d}')
assert freed == 2700 and refill == 2700 and net == 21

# 탐욕: 밀도 높은 것부터, 예산 넘으면 건너뛰기 (음수·궤적은 별도 처리)
def greedy():
    tok, score = MANDATORY_TOK, MANDATORY_SCORE
    # 궤적은 요약을 기본 후보로 (밀도 15.6 > 원본 5.6)
    tok += TRAJ['summary'][0]; score += TRAJ['summary'][1]
    for k in sorted(OPTS, key=lambda k: -density(*OPTS[k])):
        t, s = OPTS[k]
        if s > 0 and tok + t <= BUDGET:  # 음수 기여는 담지 않음
            tok += t; score += s
    return tok, score

g_tok, g_score = greedy()
print(f'\n탐욕(밀도순): {g_tok:,}토큰 / {g_score}점')
assert g_score == best_score  # 이번 예제에선 탐욕이 최적과 일치 (보장은 아님 — 배낭 문제)
print('✅ 이번 예제에선 탐욕 = 최적 (단, 배낭 문제 일반에서는 보장되지 않음)')

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 예산을 늘리며 얻을 수 있는 최대 점수 (전수 탐색)
def best_score_for(budget):
    b = None
    for traj in TRAJ:
        for r in range(len(keys) + 1):
            for picks in combinations(keys, r):
                tok, score = config_cost(traj, picks)
                if tok <= budget and (b is None or score > b):
                    b = score
    return b if b is not None else 0

budgets = list(range(1000, 12001, 500))
scores = [best_score_for(b) for b in budgets]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(budgets, scores, marker='o', ms=4, color='#6366f1')
ax.axvline(8000, ls='--', lw=1, color='#f43f5e', label='우리 예산 8,000')
ax.axhline(88, ls=':', lw=1, color='#94a3b8', label='최적 88점')
ax.set_xlabel('컨텍스트 예산 (토큰)')
ax.set_ylabel('얻을 수 있는 최대 점수')
ax.set_title('예산이 커질수록 최대 점수는? — 수확 체감(밀도 낮은 것만 남는다)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. 오늘의 발견

- **컨텍스트 윈도 = 유한한 책상** — 넘치면(9,600 > 8,000) 아예 못 읽는다. 최적화는 그 다음.
- **크기가 아니라 밀도** — 토큰당 값어치로 골라야 한다. 잡동사니(−3)는 자리만 먹는 게 아니라 방해가 된다.
- **요약 = 손실 압축**(5부 2장) — 그 자체론 −6이지만, 열린 자리를 밀도 높은 것으로 채우면 +21 순증.
- **검색(RAG, 7부 8장)** — 관련성 낮은 문서는 서고에 두고 필요할 때만 꺼낸다.
- 이 최적화가 곧 **배낭 문제** — 탐욕이 대체로 통하지만 보장은 아니라 전수 탐색으로 확정했다.

다음은 마지막 판정 — 11장 에이전트의 성적표(pass@k와 오차범위).
**웹 챕터로 돌아가 연습 10문제를 풀면 진도가 채워져요.**